In [20]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model named "Anjos_Chapter2"
m = gp.Model("Anjos_Chapter2-rep")

# Sets 
generators = [1, 2, 3]
time_periods = [1, 2, 3, 4, 5, 6]

# Parameters 
# Startup Costs (cU)
startup_cost = {1: 800, 2: 500, 3: 250}

# Variable Production Costs (c)
var_cost = {1: 5, 2: 15, 3: 30}

# Min and Max Power Output (P_min, P_max)
p_min_limit = {1: 80, 2: 50, 3: 30}
p_max_limit = {1: 300, 2: 200, 3: 100} 


# Startup and down limit (SU, SD)

start_up = {1: 100, 2: 70, 3: 40}
start_down = {1: 80, 2: 50, 3: 30}

# Ramp Up and Down Limits (RU, RD)
ramp_up = {1: 50, 2: 60, 3: 70} #RU1 from 50 to 60 
ramp_down = {1: 30, 2: 40, 3: 50} #RD1 from 30 to 50

#Timeup and Timedown 
time_up = {1: 3, 2: 2, 3: 1} #time unit has to stay on until can be allowed to shutdown
time_down = {1: 2, 2: 2, 3: 2}

# Demand (D) and Reserve (R)
demand = {1: 240, 2: 250, 3: 200, 4: 170, 5: 230, 6: 190} #replace t=5 for 260 for 2.7.2 original D(1) = 230
reserve = {t: 10 for t in time_periods}


# v[j,t]: 1 if unit j is on at time t, 0 otherwise
v = m.addVars(generators, time_periods, vtype=GRB.BINARY, name="v")

# y[j,t]: 1 if unit j starts up at time t
y = m.addVars(generators, time_periods, vtype=GRB.BINARY, name="y")

# z[j,t]: 1 if unit j shuts down at time t
z = m.addVars(generators, time_periods, vtype=GRB.BINARY, name="z")

# p[j,t]: Power generated (Continuous, must be >= 0)
p = m.addVars(generators, time_periods, lb=0, vtype=GRB.CONTINUOUS, name="p")

# Initial Conditions (State at t=0)

# 1. Initial Status (v_init): 1 = ON, 0 = OFF
# Unit 1 was ON, Units 2 & 3 were OFF
v_init = {1: 1, 2: 0, 3: 0}

# 2. Initial Power Output (p_init): in MW
# Unit 1 was producing 120 MW
p_init = {1: 120, 2: 0, 3: 0}

# 3. Initial Up-Time Counter (u_init): How many hours has the unit ALREADY been ON?
# Unit 1 has been on for 2 hours already. (Needed for Min Up Time constraint)
u_init = {1: 2, 2: 0, 3: 0}

# 4. Initial Down-Time Counter (d_init): How many hours has the unit ALREADY been OFF?
# (Not strictly needed for this specific example since only Unit 1 is on, but good practice)

d_init = {1: 0, 2: 0, 3: 0}

# d_init = {1: 0, 2: 0, 3: 2} 


#d_init = {1: 0, 2: 2, 3: 2}

In [21]:
# Objective Function (2.1)

obj_func = gp.quicksum(
    var_cost[j] * p[j, t] + startup_cost[j] * y[j, t] 
    for j in generators 
    for t in time_periods)

m.setObjective(obj_func, GRB.MINIMIZE)

 Our Objective function : $\min_{p_j(t)}\sum_{t \in T}\sum_{j \in J}{(c_j(t)p_j(t) + c_j^U(t)y_j(t)}) \qquad$                    (2.1)

In [22]:
#constraint 2.2

power_balance = {}
for t in time_periods:
    power_balance[t] = m.addConstr(
        gp.quicksum(
        p[j, t]
        for j in generators)  == demand[t],
        name=f"forecast_meeting_{t}"
        )

Constraint 2.2

$$
\sum_{j \in J} p_j(t)  = \mathbf{D}(t) \qquad \forall t \in T
$$

In [23]:
#adding maximum avaiable power- 2.3

p_max = m.addVars(generators, time_periods, lb=0, vtype=GRB.CONTINUOUS, name="p_max")

#part of Constraint 2.4?
#for t in time_periods:
    #for j in generators:
        #m.addConstr(p_max[j, t] <= p_max_limit[j] * v[j, t],
                    #name= f"max_power_limit_{j}_{t}")

#Constraint 2.3 ### DIDNT THIS WORK? - I think it did, but I added the above constraint 
# to make sure that the max power is limited by the on/off status of the generator.

#for t in time_periods:
    #m.addConstr(
        #gp.quicksum(p_max[j, t] 
            #for j in generators
            #) >= demand[t] + reserve[t],
            #name = f"max_power_reserve_{j}_{t}")

reserve_req = {}
for t in time_periods:
    reserve_req[t] = m.addConstr(
        gp.quicksum(p_max[j, t] for j in generators) >= demand[t] + reserve[t],
        name=f"reserve_req_static_{t}"
    )


Constr 2.3

$$
\sum_{j \in J} \bar{p}_j(t) \geq \mathbf{D}(t) + \mathbf{R}(t) \qquad \forall t \in T
$$

In [24]:
#Constraint 2.4  

#Look above

$$
\bar{p}_j(t) \leq \bar{P}_j \times v[j, t] \qquad \forall t \in T, \forall j \in J
$$

In [25]:
#COnstraint 2.5

for t in time_periods:
    for j in generators:
        v_prev = v_init[j] if t == 1 else v[j, t-1]
        m.addConstr(v_prev - v[j, t] + y[j, t] - z[j, t] == 0,
                    name = f"logic_binaries_{j}_{t}")

Constraint 2.5

$$
v_j(t-1) - v_j(t) + y_j(t) - z_j(t) = 0 \qquad \forall j \in J, \forall t \in T
$$

In [26]:
#Constraint 2.6 Starting-up Cost

#for t in time_periods:
    #for j in generators:
        #m.addConstr(
        #startup_cost >= 

$$
c_j^U(t) \geq \mathbf{k}_j^{\tau}\left(v_j(t) - \sum_{n=1}^{\tau}v_j(t-n) \right) \qquad \tau = 0,1,...,\tau_j^{cold} \quad \forall j \in J \quad \forall t \in T
$$

In [27]:
#Constraint 2.7 - Ramping-up constraints

for j in generators:
    for t in time_periods:
        p_prev = p_init[j] if t == 1 else p[j, t-1]
        v_prev = v_init[j] if t == 1 else v[j, t-1]
        m.addConstr(
            p[j, t] - p_prev <= ramp_up[j] * v_prev + start_up[j] * y[j, t],
            name = f"ramp_up_limit_{j}_{t}"
        )

Constraint 2.7

$$
p_j(t) - p_j(t-1) \leq R_j^U v_j(t-1) + S_j^U y_j(t) \qquad \forall j \in J \quad \forall t \in T
$$

In [28]:
#Constraint 2.8

for j in generators:
    for t in time_periods:
        p_prev = p_init[j] if t == 1 else p[j, t-1]
        m.addConstr(
            p_prev - p[j, t] <= ramp_down[j] * v[j,t] + start_down[j] * z[j, t],
            name= f"ramp_down_limit_{j}_{t}"
        )

Constraint 2.8

$$
p_j(t-1) - p_j(t) \leq R_j^D v_j(t) + S_j^D z_j(t) \qquad \forall j \in J \quad \forall t \in T
$$

In [29]:
#2.9 - Uptime constraints

total_hours = len(time_periods)

for j in generators:
    T_U = time_up[j]             # Minimum up time required (e.g., 3)
    already_on = u_init[j]      # Hours needing to be on at the start (e.g., 2)

    if v_init[j] == 1:
        remaining_needed = u_init[j]
        L_j = max(0, min(total_hours, remaining_needed))  # Unit is off, so no forced UP time, UPDATE: SWAPED THIS IN V2
    else:   
     #was remaining_needed = T_U - u_init[j] previously
        L_j = 0
    
    # The Forced Period (t = 1 to L_j)
    for t in range(1, L_j + 1):
        m.addConstr(
            v[j, t] == 1,
            name=f"forced_up_{j}_{t}"
        )
    
    # The Standard Period (t = L_j + 1 to End)
    # (t - TU + 1) is guaranteed to be >= 1.
    for t in range(L_j + 1, total_hours + 1):
        
        # Calculate the start of the window
        start_window = t - T_U + 1
        start_window = max(1, start_window)
        
        # Sum startups in the window
        recent_startups = gp.quicksum(y[j, k] for k in range(start_window, t + 1))
        
        # Add the constraint
        m.addConstr(
            v[j, t] >= recent_startups,
            name=f"min_up_time_{j}_{t}"
        )

COnstraint 2.9- Minimum uptime

$$
\sum_{k = t - T_j^U + 1}^t y_j(k) \leq v_j(t) \qquad \forall t \in [L_j + 1,...,|T|], \quad \forall j \in J
$$


In [30]:
# COnstraint 2.10 - minimum downtime

total_hours = len(time_periods)

for j in generators:
    T_D = time_down[j]             # Minimum down time required (e.g., 2)
    already_off = d_init[j]      # Hours already off at start (e.g., 0)

    if v_init[j] == 0:
        F_j = 0 #
    else:
        remaining_needed_off = already_off
        F_j = max(0, min(total_hours, remaining_needed_off))   

    
    # The Forced Period (t = 1 to F_j)
    for t in range(1, F_j + 1):
        m.addConstr(
            v[j, t] == 0,
            name=f"forced_down_{j}_{t}"
        )
    
    # The Standard Period (t = F_j + 1 to End)
    # (t - TU + 1) is guaranteed to be >= 1.
    for t in range(F_j + 1, total_hours + 1):
        
        # Calculate the start of the window
        start_window = t - T_D + 1
        start_window = max(1, start_window)
        
        # Sum shutdowns in the window
        recent_shutdowns = gp.quicksum(z[j, k] for k in range(start_window, t + 1))
        
        # Add the constraint
        m.addConstr(
            v[j, t] + recent_shutdowns <= 1,
            name=f"min_down_time_{j}_{t}"
        )

Condition 2.10 - Minimum Downtime

$$
v_j(t) + \sum_{k = t - T_j^U + 1}^t z_j(k) \leq 1 \qquad \forall t \in [F_j + 1,...,|T|], \quad \forall j \in J
$$

In [31]:
# Constraint 2.11 - Generation limit

for j in generators:
    for t in time_periods:
        m.addConstr(
            p_max[j, t] <= p_max_limit[j] * v[j, t], 
            name=f"static_max_{j}_{t}"
        )
        m.addConstr(
            p_min_limit[j] * v[j, t] <= p[j, t],
            name=f"min_power_limit_{j}_{t}" 
        )
        m.addConstr(
            p[j, t] <= p_max[j, t],
            name=f"max_power_limit_{j}_{t}"
        )
        #m.addConstr(
            #p[j, t] <= p_max[j, t],
            #name=f"gen_le_avail_{j}_{t}"
        #)
        #m.addConstrs(
            #p[j, t] <= p_max[j, t], 
            #name=f"gen_le_avail_{j}_{t}"
        #)


Constraint 2.11 - Generating limit

$$
\underline{P_j} v_j(t) \leq p_j(t) \leq \overline{p_j}(t)\leq \overline{P_j}v_j(t) \qquad \forall j \in J, \quad \forall t \in T
$$

In [32]:
# constriant 2.12

for j in generators:
    for t in time_periods:
        p_prev = p_init[j] if t == 1 else p[j, t-1]
        v_prev = v_init[j] if t == 1 else v[j, t-1] 
        m.addConstr(
            p_max[j, t] <= p_prev + ramp_up[j] * v_prev + start_up[j] * y[j, t],
            name = f"ramp_up_limit_max_{j}_{t}"
            )

Constraint 2.12 - max power outage at t

$$
\overline{p_j}(t) \leq p_j(t-1) + R_j^U v_j(t-1) + S_j^U y_j(t) \qquad \forall j \in J, \quad \forall t \in T
$$

In [33]:
# COnstraint 2.13 - limiting maximum power outage

for j in generators:
    for t in time_periods:

        z_next = z[j, t+1] if t < 6 else 0#m.addConstr(p_max[j, t] <= p_max_limit[j] * v[j, t],name=f"max_power_outage_t6_{j}_{t}")

        m.addConstr(
            p_max[j, t] <= p_max_limit[j] * v[j, t] - p_max_limit[j] * z_next + z_next * start_down[j],
            name = f"max_power_outage_{j}_{t}"
        )

Constraint 2.13- power outage limit

$$
\overline{p_j}(t) \leq \overline{P_j}\left[ v_j(t) - z_j(t + 1) \right] + z_j(t+1) S_j^D \qquad \forall j \in J, \quad \forall t \in T
$$

In [34]:
# Optimisation

m.optimize()



Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i5-1250P, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 192 rows, 90 columns and 563 nonzeros (Min)
Model fingerprint: 0x85b48204
Model has 36 linear objective coefficients
Variable types: 36 continuous, 54 integer (54 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [5e+00, 8e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+02]

Found heuristic solution: objective 20600.000000
Presolve removed 85 rows and 37 columns
Presolve time: 0.01s
Presolved: 107 rows, 53 columns, 356 nonzeros
Variable types: 30 continuous, 23 integer (23 binary)

Root relaxation: objective 9.629979e+03, 39 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth Int

In [35]:
# Dual Variables too hard to extract, thus we can just see by marginal cost of units what duals is. 

In [36]:
print("\nOptimal Solution: ")
print(f"Optimal Objective Value: £{m.ObjVal:.2f}")
print("-" * 80)
print(f"{'Hour':<5} | {'Unit':<5} | {'Status (v)':<10} | {'Power (p)':<10} | {'Startup (y)':<10} | {'Shutdown (z)':<10}")
print("-" * 80)

for t in time_periods:
    for j in generators:
        v_val = int(v[j, t].X)
        p_val = p[j, t].X
        y_val = int(y[j, t].X)
        z_val = int(z[j, t].X)
        
        if v_val == 1 or p_val > 0.1 or y_val == 1 or z_val == 1:
            print(f"{t:<5} | {j:<5} | {v_val:<10} | {p_val:<10.2f} | {y_val:<10} | {z_val:<10}")


Optimal Solution: 
Optimal Objective Value: £9900.00
--------------------------------------------------------------------------------
Hour  | Unit  | Status (v) | Power (p)  | Startup (y) | Shutdown (z)
--------------------------------------------------------------------------------
1     | 1     | 1          | 160.00     | 0          | 0         
1     | 2     | 1          | 50.00      | 1          | 0         
1     | 3     | 1          | 30.00      | 1          | 0         
2     | 1     | 1          | 200.00     | 0          | 0         
2     | 2     | 1          | 50.00      | 0          | 0         
2     | 3     | 0          | 0.00       | 0          | 1         
3     | 1     | 1          | 200.00     | 0          | 0         
3     | 2     | 0          | 0.00       | 0          | 1         
4     | 1     | 1          | 170.00     | 0          | 0         
5     | 1     | 1          | 200.00     | 0          | 0         
5     | 3     | 1          | 30.00      | 1          | 

In [37]:
if m.status == GRB.INFEASIBLE:
    print("Model is Infeasible. Computing IIS to find the conflict...")
    m.computeIIS()
    m.write("conflict.ilp")
    print("Conflict saved to 'conflict.ilp'. Open this file to see the bad constraints.")
    
    # Optional: Print the conflicting constraints directly to the screen
    for c in m.getConstrs():
        if c.IISConstr:
            print(f"Conflict found in: {c.ConstrName}")


In [38]:
#!jupyter nbconvert --to pdf Basic_model_V2.ipynb